In [ ]:
!pip install --upgrade transformers datasets --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.

In [ ]:
import pandas as pd
from datasets import Dataset

# Load the new Gen Z slang dataset (5000 lines)
df = pd.read_csv("/content/genz_slang_5000.csv")

# Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df.rename(columns={"sentence": "text"}))
dataset = dataset.select(range(1000))

In [ ]:
from transformers import AutoTokenizer

model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Required for GPT-Neo

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Embedding(50257, 768)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,  # 🚀 1 epoch for speed
    per_device_train_batch_size=2,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",     # ✅ disables wandb
    fp16=True             # ⚡ if GPU supports it
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


In [ ]:
trainer.train()


Step,Training Loss
10,3.302600
20,0.892100
30,0.653100
40,0.539900
50,0.490000
60,0.365900
70,0.352900
80,0.319100
90,0.279700
100,0.270000


TrainOutput(global_step=500, training_loss=0.28375096106529235, metrics={'train_runtime': 1584.2305, 'train_samples_per_second': 0.631, 'train_steps_per_second': 0.316, 'total_flos': 32650887168000.0, 'train_loss': 0.28375096106529235, 'epoch': 1.0})

In [ ]:
model.save_pretrained("./genz-gpt-neo")
tokenizer.save_pretrained("./genz-gpt-neo")


('./genz-gpt-neo/tokenizer_config.json',
 './genz-gpt-neo/special_tokens_map.json',
 './genz-gpt-neo/vocab.json',
 './genz-gpt-neo/merges.txt',
 './genz-gpt-neo/added_tokens.json',
 './genz-gpt-neo/tokenizer.json')

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="./genz-gpt-neo", tokenizer=tokenizer)

# Try it out
result = generator("this party is so", max_new_tokens=20, do_sample=True, top_k=50, top_p=0.95)
print(result[0]["generated_text"])


Device set to use cpu


this party is so zesty that no cap


hybrid moodel

In [ ]:
!pip install transformers --quiet



In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch

# Load BERT (for context understanding)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Load fine-tuned GPT-Neo from local directory
gpt_tokenizer = AutoTokenizer.from_pretrained("/content/genz-gpt-neo", local_files_only=True)
gpt_model = AutoModelForCausalLM.from_pretrained("/content/genz-gpt-neo", local_files_only=True)
gpt_model.eval()
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
def generate_with_hybrid_bert_gptneo(user_input, max_new_tokens=25):
    # Step 1: BERT encodes the input (this part could be extended in deeper models)
    bert_inputs = bert_tokenizer(user_input, return_tensors="pt")
    with torch.no_grad():
        _ = bert_model(**bert_inputs)

    # Step 2: GPT-Neo generates continuation based on raw input
    gpt_inputs = gpt_tokenizer(user_input, return_tensors="pt")
    gpt_output = gpt_model.generate(
        **gpt_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=gpt_tokenizer.eos_token_id
    )

    return gpt_tokenizer.decode(gpt_output[0], skip_special_tokens=True)


In [ ]:
print(generate_with_hybrid_bert_gptneo("yo that fit is"))
print(generate_with_hybrid_bert_gptneo("this song is"))
print(generate_with_hybrid_bert_gptneo("bruh she really"))


yo that fit is fire fr fr
this song is acting goated fr fr
bruh she really drippin no cap


In [ ]:
# Define simple GPT-only generation function
def generate_with_gptneo(prompt, max_new_tokens=25):
    inputs = gpt_tokenizer(prompt, return_tensors="pt")
    outputs = gpt_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=gpt_tokenizer.eos_token_id
    )
    return gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Compare both models on a prompt
prompt = "she ghosted him and"
output_gpt = generate_with_gptneo(prompt)
output_hybrid = generate_with_hybrid_bert_gptneo(prompt)

print("🧠 Prompt:", prompt)
print("🤖 GPT-Neo Output:", output_gpt)
print("🔀 Hybrid Output:", output_hybrid)


🧠 Prompt: she ghosted him and
🤖 GPT-Neo Output: she ghosted him and it was mid 🔥
🔀 Hybrid Output: she ghosted him and it was zesty 🫠
